In [1]:
import pandas as pd
import lizard
import os

PROCESSED_DIR = r"C:\Users\HP\Desktop\thesis_preprocessing\data\processed"

FILES =[
    "01_Original_12k.csv",
    "02_Distractor_A_Swapped_12k.csv",
    "03_Distractor_B_Shuffled_12k.csv"
]

ext_map = {
    "java": ".java",
    "python": ".py",
    "go": ".go",
    "ruby": ".rb",
    "php": ".php",
    "javascript": ".js"
}

def calculate_adak(row):
    code = str(row['original_code'])
    comment = str(row['comment'])
    lang = str(row['language']).lower()
    
    # Calculate MCV
    comment_lines = len(comment.splitlines())
    mcv = comment_lines * 0.8
    
    # Calculate SFV
    ext = ext_map.get(lang, ".txt")
    dummy_filename = f"temp_file{ext}"
    
    try:
        analysis = lizard.analyze_file.analyze_source_code(dummy_filename, code)
        token_count = analysis.token_count
    except:
        token_count = 0
        
    # Fallback for unparsable fragments
    if token_count == 0:
        token_count = len(code.split())
        if token_count == 0:
            token_count = 1 
            
    sfv = token_count * 0.3
    
    # Calculate Adak Index
    adak_index = ((100 * mcv) / sfv) - 100
    
    return pd.Series([mcv, sfv, adak_index])

In [2]:
for file_name in FILES:
    print(f"Processing: {file_name}")
    file_path = os.path.join(PROCESSED_DIR, file_name)
    
    # Load dataset
    df_temp = pd.read_csv(file_path)
    
    # Apply Adak calculation
    df_temp[['mcv', 'sfv', 'adak_index']] = df_temp.apply(calculate_adak, axis=1)
    
    # Save back to the same file with new columns
    df_temp.to_csv(file_path, index=False)

print("Adak calculation complete for all three datasets.")

Processing: 01_Original_12k.csv
Processing: 02_Distractor_A_Swapped_12k.csv
Processing: 03_Distractor_B_Shuffled_12k.csv
Adak calculation complete for all three datasets.


In [3]:
# Verify outputs for the first file
sample_df = pd.read_csv(os.path.join(PROCESSED_DIR, FILES[0]))
print("Verification of Original Dataset Columns:")
print(sample_df.columns.tolist())
print("\nAdak Statistics:")
print(sample_df[['mcv', 'sfv', 'adak_index']].describe())

Verification of Original Dataset Columns:
['language', 'repo', 'path', 'func_name', 'original_code', 'comment', 'code_token_length', 'label', 'mcv', 'sfv', 'adak_index']

Adak Statistics:
                mcv           sfv    adak_index
count  12000.000000  12000.000000  12000.000000
mean       4.198867     23.137350    144.968467
std        6.680388     24.434556    665.545901
min        0.800000      0.300000    -99.591628
25%        0.800000      8.700000    -94.202899
50%        2.400000     15.900000    -85.454545
75%        4.800000     29.400000    -44.654088
max      226.400000    195.900000  16166.666667
